#### Resources
1. Pyspark tutorial : https://www.kaggle.com/code/nilaychauhan/pyspark-tutorial-for-beginners
2. Advanced Pyspark for Exploratory Data Analysis : https://www.kaggle.com/code/tientd95/advanced-pyspark-for-exploratory-data-analysis
3. PySpark for Statistical Inference : https://www.kaggle.com/code/tientd95/pyspark-for-statistical-inference
4. PySpark for Data Science : https://www.kaggle.com/code/tientd95/pyspark-for-data-science

In [15]:
# Import other modules not related to PySpark
# import os
import sys
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
import math
from IPython.core.interactiveshell import InteractiveShell
from datetime import *
import statistics as stats
# This helps auto print out the items without explixitly using 'print'
InteractiveShell.ast_node_interactivity = "all" 
%matplotlib inline

In [16]:
# Import PySpark related modules
import pyspark
from pyspark.rdd import RDD
from pyspark.sql import Row
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql import functions
from pyspark.sql.functions import lit, desc, col, size, array_contains\
, isnan, udf, hour, array_min, array_max, countDistinct
from pyspark.sql.types import *

In [1]:
import os

In [2]:
DIR = r'/kaggle/input/playground-series-s6e1'
TEST = os.path.join(DIR, r'test.csv')
TRAIN = os.path.join(DIR, r'train.csv')

In [3]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

Create a **SparkContext** object. With the SparkContext, you can input a dataset and parallelize the data across a cluster.

In [4]:
spark = SparkSession \
    .builder \
    .appName("Pred_Student_Scores_EDA") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/23 05:22:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.sparkContext.getConf().getAll()

[('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.app.startTime', '1769145738175'),
 ('spark.executor.id', 'driver'),
 ('spark.app.submitTime', 

In [6]:
spark

### Alternate method to initialize a pyspark session:

```python
MAX_MEMORY = '15G'
# Initialize a spark session.
conf = pyspark.SparkConf().setMaster("local[*]") \
        .set('spark.executor.heartbeatInterval', 10000) \
        .set('spark.network.timeout', 10000) \
        .set("spark.core.connection.ack.wait.timeout", "3600") \
        .set("spark.executor.memory", MAX_MEMORY) \
        .set("spark.driver.memory", MAX_MEMORY)
def init_spark():
    spark = SparkSession \
        .builder \
        .appName("Pyspark guide") \
        .config(conf=conf) \
        .getOrCreate()
    return spark

spark = init_spark()
filename_data = '../input/fitrec-dataset/endomondoHR.json'
# Load the main data set into pyspark data frame 
df = spark.read.json(filename_data, mode="DROPMALFORMED")
print('Data frame type: ' + str(type(df)))
```

### Reading the Dataframe using spark

By default, Spark assumes there is no header and all columns are of type StringType, named _c0, _c1, etc.. 

### Common Options
You can customize the reading behavior using options for more robust data loading: 

Option | Default | Description | Example |
-------| ------- | ----------- | ------- |
header	| `False`	| Set to `True` if the first row is the column header. | `header=True` |
inferSchema	| `False` | Set to `True` to automatically determine column data types (involves an extra pass over the data, which can be slow for large files).	| `inferSchema=True` |
sep	| `,`	| Specifies the column delimiter if it's not a comma (e.g., `'\\t'` for tabs, \`'	'\` for pipes). | |
mode | `PERMISSIVE` | Controls handling of corrupt records: `PERMISSIVE` (default), `DROPMALFORMED` (drops rows with corrupt data), or `FAILFAST` (aborts the task).	| `mode="DROPMALFORMED"` |

In [7]:
# Reading the csv file into a dataframe
df = spark.read.csv(TEST, header=True, inferSchema=True)

# Alternate code:
# df_options_alt = (
#     spark.read.format("csv")
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .load("path/to/yourfile.csv")
# )

#### Reading Multiple Files
From a single directory: Pass the path to the directory, and PySpark will read all CSV files within it.
```python
df_dir = spark.read.csv("path/to/directory/")
```
From multiple specific paths: Pass a list of file paths.
```python
paths = ["path/to/file1.csv", "path/to/file2.csv"]
df_multiple = spark.read.csv(paths, header=True, inferSchema=True)
```

#### Custom Schema in Pyspark

In PySpark, you specify a custom schema using the `StructType` class, which is a collection of `StructField` objects. This allows you to explicitly define column names, data types, and nullability, rather than relying on automatic schema inference. 

##### Steps to Define a Custom Schema
1. **Import necessary types:** Import `StructType`, `StructField`, and the specific data types you need (e.g., `StringType`, `IntegerType`, `DateType`) from `pyspark.sql.types`.
2. **Define the schema:** Create an instance of `StructType` containing a list of `StructField` instances. Each `StructField` takes the column name, data type, and a boolean indicating nullability (whether the column can contain `None` values).
3. **Apply the schema:**
    * When creating a DataFrame from a local collection, pass the schema to the `spark.createDataFrame()` method.
    * When reading data from a file (like CSV or JSON), pass the schema to the reader method (e.g., `.schema()`).


#### Example: Defining and Applying a Schema
This example demonstrates how to define a custom schema and use it when creating a DataFrame from a list of data. 
For an example of defining a schema and applying it when creating a DataFrame from data, you can refer to medium.com. 

**Example: Applying Schema When Reading a File**
A custom schema can be applied when reading data from files like CSV using the `.schema()` option, which offers better performance than schema inference. 

```python
# Assuming you have a custom_schema defined
df_csv = (spark.read
    .format("csv")
    .option("header", True) # Assuming your CSV has a header
    .schema(custom_schema) # Specify the custom schema
    .load("path/to/your/data.csv")
)

df_csv.printSchema()

```

**Complex and Nested Schemas**
For complex data structures such as nested JSON, a schema can be defined by embedding `StructType` within a `StructField`. An example of a nested schema definition is available at [medium.com](https://medium.com/@softwareprocesspains2023/pyspark-how-to-define-a-custom-nested-schema-for-a-dataframe-and-how-its-displayed-in-a-hive-1c054f632ff4).


Also refer to [stockoverflow.com](https://stackoverflow.com/questions/57901493/pyspark-defining-custom-schema-for-a-dataframe#:~:text=Related,schema%20of%20a%20df%20pyspark).


In [8]:
# Displaying the dataframe
df.show()

+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|    id|age|gender| course|study_hours|class_attendance|internet_access|sleep_hours|sleep_quality| study_method|facility_rating|exam_difficulty|
+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|630000| 24| other|     ba|       6.85|            65.2|            yes|        5.2|         poor|  group study|           high|           easy|
|630001| 18|  male|diploma|       6.61|            45.0|             no|        9.3|         poor|     coaching|            low|           easy|
|630002| 24|female| b.tech|        6.6|            98.5|            yes|        6.2|         good|  group study|         medium|       moderate|
|630003| 24|  male|diploma|       3.03|            66.3|            yes|        5.7|      average|        mixed|         medium|  

In [17]:
# Displaying the dataframe's schema
print('Data overview')
df.printSchema()

Data overview
root
 |-- id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- course: string (nullable = true)
 |-- study_hours: double (nullable = true)
 |-- class_attendance: double (nullable = true)
 |-- internet_access: string (nullable = true)
 |-- sleep_hours: double (nullable = true)
 |-- sleep_quality: string (nullable = true)
 |-- study_method: string (nullable = true)
 |-- facility_rating: string (nullable = true)
 |-- exam_difficulty: string (nullable = true)



In [18]:
print('Columns overview')
pd.DataFrame(df.dtypes, columns = ['Column Name','Data type'])

Columns overview


,Column Name,Data type
0,id,int
1,age,int
2,gender,string
3,course,string
4,study_hours,double
5,class_attendance,double
6,internet_access,string
7,sleep_hours,double
8,sleep_quality,string
9,study_method,string


In [10]:
# df.head() in pyspark retrieves the first row of a DataFrame as a single row object, 
# unlike in pandas - where the first 5 rows of the df are retrieved as a df
df.head()

Row(id=630000, age=24, gender='other', course='ba', study_hours=6.85, class_attendance=65.2, internet_access='yes', sleep_hours=5.2, sleep_quality='poor', study_method='group study', facility_rating='high', exam_difficulty='easy')

In [11]:
# df.head(n) in pyspark retrieves the first n rows of the DataFrame as a list of single row objects corresponding
# to each row of the DataFrame
df.head(5)

[Row(id=630000, age=24, gender='other', course='ba', study_hours=6.85, class_attendance=65.2, internet_access='yes', sleep_hours=5.2, sleep_quality='poor', study_method='group study', facility_rating='high', exam_difficulty='easy'),
 Row(id=630001, age=18, gender='male', course='diploma', study_hours=6.61, class_attendance=45.0, internet_access='no', sleep_hours=9.3, sleep_quality='poor', study_method='coaching', facility_rating='low', exam_difficulty='easy'),
 Row(id=630002, age=24, gender='female', course='b.tech', study_hours=6.6, class_attendance=98.5, internet_access='yes', sleep_hours=6.2, sleep_quality='good', study_method='group study', facility_rating='medium', exam_difficulty='moderate'),
 Row(id=630003, age=24, gender='male', course='diploma', study_hours=3.03, class_attendance=66.3, internet_access='yes', sleep_hours=5.7, sleep_quality='average', study_method='mixed', facility_rating='medium', exam_difficulty='moderate'),
 Row(id=630004, age=20, gender='female', course='b.t

```python
df.head()
```
* **Action, not Transformation:** `head()` is an action, meaning it triggers the execution of all preceding transformations and collects the data to the driver's memory.
* **Memory Warning:** It should only be used when you expect the result to be small. Passing a very large n to a huge DataFrame can cause an `OutOfMemoryError` on the driver node.
* **Alternative for Display:** For simply viewing data in a tabular format in the console, `df.show()` or `df.show(n)` is generally preferred, as it handles formatting and truncation more efficiently for large datasets.
* **Alternative for large n:** If you need a large number of rows for further processing (not just displaying), consider using `df.limit(n)`, which returns a new DataFrame (a transformation, not an action). 

In [13]:
df.show(5) # by default shows 20 rows

+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+------------+---------------+---------------+
|    id|age|gender| course|study_hours|class_attendance|internet_access|sleep_hours|sleep_quality|study_method|facility_rating|exam_difficulty|
+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+------------+---------------+---------------+
|630000| 24| other|     ba|       6.85|            65.2|            yes|        5.2|         poor| group study|           high|           easy|
|630001| 18|  male|diploma|       6.61|            45.0|             no|        9.3|         poor|    coaching|            low|           easy|
|630002| 24|female| b.tech|        6.6|            98.5|            yes|        6.2|         good| group study|         medium|       moderate|
|630003| 24|  male|diploma|       3.03|            66.3|            yes|        5.7|      average|       mixed|         medium|       mo

# 2. Exploratory Data Analysis

## 2.1. Basic Summary / Description of columns

In PySpark, the `describe()` function is used to compute basic summary statistics for columns in a DataFrame, which is an essential part of exploratory data analysis (EDA). 

**Functionality :**
The `describe()` method analyzes both numeric and string columns: 
* **For numerical columns:** It computes the `count` (number of non-null entries), `mean`, `stddev` (standard deviation), `min`, and `max` values.
* **For string/categorical columns:** It computes the `count`, `unique` (number of unique values), `top` (most frequent value), and `freq` (frequency of the top value).
* If no columns are specified, it computes these statistics for all numerical and string columns in the DataFrame. 

In [19]:
print('Data frame describe (string and numeric columns only):')
df.describe().toPandas()

Data frame describe (string and numeric columns only):


26/01/23 05:58:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,summary,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,count,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000
1,mean,764999.5,20.544137037037036,None,None,4.0038779962962066,71.98250877777949,None,7.072069925926047,None,None,None,None
2,stddev,77942.43067803312,2.2604515515553794,None,None,2.3577413963576217,17.41469469347976,None,1.7455134105990133,None,None,None,None
3,min,630000,17,female,b.com,0.08,40.6,no,4.1,average,coaching,high,easy
4,max,899999,24,other,diploma,7.91,99.4,yes,9.9,poor,self-study,medium,moderate


In [24]:
# Describing specific columns
df.describe("age").show()

+-------+------------------+
|summary|               age|
+-------+------------------+
|  count|            270000|
|   mean|20.544137037037036|
| stddev|2.2604515515553794|
|    min|                17|
|    max|                24|
+-------+------------------+



`describe()` vs. `summary()`
PySpark also offers a `summary()` method, which is generally recommended for more detailed analysis. 
* `describe()` provides the basic set of statistics (count, mean, stddev, min, max).
* `summary()` includes all the statistics from `describe()`, plus approximate quartiles (25%, 50%, and 75% percentiles) by default, and allows you to specify custom statistics as arguments. 

In [29]:
# Using the summary() method for expanded statistics
df.summary("count", "min", "max", "mean", "stddev", "25%", "75%").show()

+-------+-----------------+------------------+------+-------+------------------+-----------------+---------------+------------------+-------------+------------+---------------+---------------+
|summary|               id|               age|gender| course|       study_hours| class_attendance|internet_access|       sleep_hours|sleep_quality|study_method|facility_rating|exam_difficulty|
+-------+-----------------+------------------+------+-------+------------------+-----------------+---------------+------------------+-------------+------------+---------------+---------------+
|  count|           270000|            270000|270000| 270000|            270000|           270000|         270000|            270000|       270000|      270000|         270000|         270000|
|    min|           630000|                17|female|  b.com|              0.08|             40.6|             no|               4.1|      average|    coaching|           high|           easy|
|    max|           899999|        

In [27]:
df.summary().toPandas()

,summary,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,count,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000
1,mean,764999.5,20.544137037037036,None,None,4.0038779962962066,71.98250877777949,None,7.072069925926047,None,None,None,None
2,stddev,77942.43067803312,2.2604515515553794,None,None,2.3577413963576217,17.41469469347976,None,1.7455134105990133,None,None,None,None
3,min,630000,17,female,b.com,0.08,40.6,no,4.1,average,coaching,high,easy
4,25%,697502,19,None,None,1.98,57.0,None,5.6,None,None,None,None
5,50%,764995,21,None,None,4.01,72.6,None,7.1,None,None,None,None
6,75%,832497,23,None,None,6.05,87.2,None,8.6,None,None,None,None
7,max,899999,24,other,diploma,7.91,99.4,yes,9.9,poor,self-study,medium,moderate


**Summary of specific statistics of all columns**

In [32]:
# Compute summary specific statistics  
print("Summary for 'age' and 'gender' columns using summary():")
df.summary("count", "mean").toPandas()

Summary for 'age' and 'gender' columns using summary():


,summary,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,count,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000,270000
1,mean,764999.5,20.544137037037036,None,None,4.0038779962962066,71.98250877777949,None,7.072069925926047,None,None,None,None


**Summary of specific statistics of specific columns**

In [33]:
# Compute summary statistics by selecting columns first
print("Count and Mean summary 'age' column:")
df.select("age").summary("count", "mean").show()

Count and Mean summary 'age' column:
+-------+------------------+
|summary|               age|
+-------+------------------+
|  count|            270000|
|   mean|20.544137037037036|
+-------+------------------+



## 2.2. Detect missing values and abnormal zeros

After having a first sight of the columns, the first thing we should check is if the data set having any missing value.


* For string columns, we check for `None` and `null`
* For numeric columns, we check for `zeroes` and `NaN`
* For array type columns, we check if the array contain `zeroes` or `NaN`


In [36]:
print('Columns overview')
columns_overview_df = pd.DataFrame(df.dtypes, columns = ['Column Name','Data type'])
columns_overview_df

Columns overview


,Column Name,Data type
0,id,int
1,age,int
2,gender,string
3,course,string
4,study_hours,double
5,class_attendance,double
6,internet_access,string
7,sleep_hours,double
8,sleep_quality,string
9,study_method,string


In [42]:
columns_overview_df.iloc[:,1].unique()

array(['int', 'string', 'double'], dtype=object)

In [53]:
numeric_cols_lst = columns_overview_df[columns_overview_df.iloc[:,1].isin(['int','double'])].iloc[:,0].tolist()
string_cols_lst = columns_overview_df[columns_overview_df.iloc[:,1] == 'string'].iloc[:,0].tolist()

* The `col()` function in PySpark is a function within the `pyspark.sql.functions` module that converts a column name string into a `pyspark.sql.Column` object.
* **Purpose:** The primary use of `col()` is to explicitly reference a column within an expression, especially in scenarios where you need *to distinguish a column name from a string literal*.
* **Immutability:** *PySpark DataFrames are immutable*. Operations involving `col()` return a new DataFrame with the applied changes, leaving the original DataFrame untouched.

In [61]:
df.filter(col("age") > 18).show()

+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|    id|age|gender| course|study_hours|class_attendance|internet_access|sleep_hours|sleep_quality| study_method|facility_rating|exam_difficulty|
+------+---+------+-------+-----------+----------------+---------------+-----------+-------------+-------------+---------------+---------------+
|630000| 24| other|     ba|       6.85|            65.2|            yes|        5.2|         poor|  group study|           high|           easy|
|630002| 24|female| b.tech|        6.6|            98.5|            yes|        6.2|         good|  group study|         medium|       moderate|
|630003| 24|  male|diploma|       3.03|            66.3|            yes|        5.7|      average|        mixed|         medium|       moderate|
|630004| 20|female| b.tech|       2.03|            42.4|            yes|        9.2|      average|     coaching|            low|  

The `pyspark.sql.Column.eqNullSafe()` function in PySpark performs an equality test that is safe for `NULL` values. Unlike standard equality operators (`==` or `=`), `eqNullSafe()` returns `True` if both values are `NULL` (or `None` in Python terms), and `False` otherwise, while still functioning as a normal equality check for non-null values. 

* In standard SQL/PySpark equality checks, `NULL == NULL` evaluates to `NULL` (which behaves as `False` in conditional filtering), not `True`. `eqNullSafe()` changes this behavior. 

Condition | Standard Equality (`==`) | `eqNullSafe()` |
--------- | ---------------------- | ------------ |
Non-null == Non-null | `True` (if values match) | `True` (if values match) |
Non-null == `NULL` | `NULL` (Falsey) | `False` |
`NULL` == Non-null | `NULL` (Falsey) | `False` |
`NULL` == `NULL` | `NULL` (Falsey) | `True` |

In [65]:
# numeric_cols_lst, string_cols_lst
# missing_values = {}
for column in df.columns:
    if column in numeric_cols_lst:
        # missing

id
age
study_hours
class_attendance
sleep_hours
